<a href="https://colab.research.google.com/github/sanjanapstech-ui/AIRWISE-AI-driven-solution-for-pollution-control-and-prediction/blob/main/major_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **API LOADING**

World Air Quality Index project api

In [ ]:
import requests
from datetime import datetime

# 1. Your WAQI API Token
TOKEN = "99e20b8412b799bbca76b7e965c87313a55280a6"

# 2. Coordinates for Karnataka bounding box (South-West to North-East)
# Format: lat1, lon1, lat2, lon2
LAT_MIN, LON_MIN = 11.5, 74.0
LAT_MAX, LON_MAX = 18.5, 78.5
BOUNDS = f"{LAT_MIN},{LON_MIN},{LAT_MAX},{LON_MAX}"

# 3. API URL for map bounds
url = f"https://api.waqi.info/map/bounds/?latlng={BOUNDS}&token={TOKEN}"

response = requests.get(url)
data = response.json()

if data['status'] == 'ok':
    stations = data['data']
    total_records = len(stations)

    # Extract update times and find the most recent one
    # Note: AQICN map bounds data usually contains station-level summaries
    update_times = []
    for s in stations:
        # Some stations might not have a timestamp in the map bounds summary
        # so we fetch the most recent one we can find
        if 'utime' in s: # Check for unix timestamp directly
            update_times.append(s['utime'])
        elif 'station' in s and 'time' in s['station']: # Check for nested string timestamp
            try:
                # Parse the datetime string, e.g., '2026-04-30T01:30:00+09:00'
                dt_object = datetime.fromisoformat(s['station']['time'])
                update_times.append(dt_object.timestamp())
            except ValueError:
                # Handle cases where datetime string format might differ or be invalid
                pass

    # Convert latest timestamp to readable date
    if update_times:
        latest_ts = max(update_times)
        last_update = datetime.fromtimestamp(latest_ts).strftime('%Y-%m-%d %H:%M:%S')
    else:
        last_update = "Unknown (No timestamp provided in summary)"

    print(f"Region: Karnataka")
    print(f"Total Reporting Stations: {total_records}")
    print(f"Latest Record Update: {last_update}")
else:
    print("Error: Could not retrieve data for the specified region.")

# **Extract Karnataka Cities from API Data**

In [ ]:
import re

# Extract unique district/city names
districts = set()
for s in stations:
    if 'station' in s and 'name' in s['station']:
        station_name = s['station']['name']

        # Split the station name by comma and take the second part as the district/city
        parts = station_name.split(',')
        if len(parts) > 1:
            district = parts[1].strip()
            districts.add(district)
        else:
            # If no comma, consider the whole name as the district or handle as needed
            districts.add(station_name.strip())

print(f"Total unique districts found: {len(districts)}")
print("Extracted Districts:")
for district in sorted(list(districts)):
    print(f"- {district}")

# **Filter Selected Cities for Analysis**

In [ ]:
selected_cities = {
    "Bagalkot",
    "Belgaum",
    "Bengaluru",
    "Chamarajanagar",
    "Chikkaballapur",
    "Chikkamagaluru",
    "Davanagere",
    "Dharwad",
    "Hubballi",
    "Kalaburagi",
    "Madikeri",
    "Mangalore",
    "Mysuru",
    "Ramanagara",
    "Tumakuru",
    "Yadgir"
}

filtered_stations = []

gov_keywords = ["CPCB", "KSPCB", "STATE POLLUTION", "BOARD", "MINISTRY"]

for s in stations:
    if 'station' in s and 'name' in s['station']:
        station_name = s['station']['name']

        # First, check if the station name contains any government keywords
        is_official = any(keyword in station_name.upper() for keyword in gov_keywords)

        if is_official:
            parts = station_name.split(',')
            if len(parts) > 1:
                district = parts[1].strip()
                if district in selected_cities:
                    filtered_stations.append(s)
            else:
                # If no comma, consider the whole name as the district
                if station_name.strip() in selected_cities:
                    filtered_stations.append(s)

print(f"Total filtered stations for selected cities (and official sources): {len(filtered_stations)}")
# Display some of the filtered stations for verification
# for s in filtered_stations[:5]:
#     print(s['station']['name'], s['aqi'])

# **City Coordinates Mapping for Visualization**

In [ ]:
city_coords = {
    "Bagalkot": [16.1725, 75.6550],
    "Belagavi": [15.8497, 74.4977],
    "Bengaluru": [12.9716, 77.5946],
    "Chamarajanagar": [11.9231, 76.9395],
    "Chikkaballapur": [13.4350, 77.7315],
    "Chikkamagaluru": [13.3161, 75.7720],
    "Davanagere": [14.4644, 75.9218],
    "Dharwad": [15.4589, 75.0078],
    "Hubli": [15.3647, 75.1240],
    "Kalaburagi": [17.3297, 76.8343],
    "Madikeri": [12.4244, 75.7382],
    "Mangaluru": [12.9141, 74.8560],
    "Mysuru": [12.2958, 76.6394],
    "Ramanagara": [12.7200, 77.2800],
    "Tumakuru": [13.3409, 77.1010],
    "Yadgir": [16.7704, 77.1378]
}

In [ ]:
district_station_ids = {
    "Bagalkot": "@14154", # Vidayagiri, Bagalkot - KSPCB
    "Belgaum": "@14155",  # Ramteerth Nagar, Belgaum - KSPCB (Note: city_coords uses 'Belagavi')
    "Bengaluru": "@8190", # BTM Layout, Bengaluru - CPCB
    "Chamarajanagar": "@14157", # Urban, Chamarajanagar - KSPCB
    "Chikkaballapur": "@14158", # Chikkaballapur Rural - KSPCB
    "Chikkamagaluru": "@14159", # Kalyana Nagara, Chikkamagaluru - KSPCB
    "Davanagere": "@14160", # Devaraj Urs Badavane - KSPCB
    "Dharwad": "@14161",  # Kalabhavan, Dharwad - KSPCB
    "Hubballi": "@14162", # Deshpande Nagar, Hubballi - KSPCB (Note: selected_cities uses 'Hubballi', city_coords uses 'Hubli')
    "Kalaburagi": "@14163", # PDA College of Engineering - KSPCB
    "Madikeri": "@14164",  # Madikeri Rural, Kodagu - KSPCB
    "Mangalore": "@14165", # Baikampady, Mangalore - KSPCB (Note: city_coords uses 'Mangaluru')
    "Mysuru": "@14166",    # Hebbal, Mysuru - KSPCB
    "Ramanagara": "@14167", # Ramanagara Rural - KSPCB
    "Tumakuru": "@14168",  # Kyathasandra, Tumakuru - KSPCB
    "Yadgir": "@14169"     # Yadgir Rural - KSPCB
}

# **Interactive Dropdown for City Selection**

In [ ]:
from ipywidgets import Dropdown
from IPython.display import display

# Ensure that dropdown options are valid keys in city_coords
valid_cities_for_dropdown = [city for city in selected_cities if city in city_coords]

dropdown = Dropdown(
    options=sorted(valid_cities_for_dropdown),
    description='City:',
)

display(dropdown)

# **AQI Sub-Index Calculation using CPCB Breakpoints**

In [ ]:
def calculate_sub_index(C, breakpoints):
    for bp in breakpoints:
      # Finds the correct range (breakpoint) where pollutant value C falls
        if bp["B_lo"] <= C <= bp["B_hi"]:
            I_lo = bp["I_lo"]
            I_hi = bp["I_hi"]
            B_lo = bp["B_lo"]
            B_hi = bp["B_hi"]

            return ((I_hi - I_lo) / (B_hi - B_lo)) * (C - B_lo) + I_lo
    return None


breakpoints = {
    "PM2.5": [
        {"B_lo": 0, "B_hi": 30, "I_lo": 0, "I_hi": 50},
        {"B_lo": 31, "B_hi": 60, "I_lo": 51, "I_hi": 100},
        {"B_lo": 61, "B_hi": 90, "I_lo": 101, "I_hi": 200},
        {"B_lo": 91, "B_hi": 120, "I_lo": 201, "I_hi": 300},
        {"B_lo": 121, "B_hi": 250, "I_lo": 301, "I_hi": 400},
        {"B_lo": 251, "B_hi": 500, "I_lo": 401, "I_hi": 500},
    ],
    "PM10": [
        {"B_lo": 0, "B_hi": 50, "I_lo": 0, "I_hi": 50},
        {"B_lo": 51, "B_hi": 100, "I_lo": 51, "I_hi": 100},
        {"B_lo": 101, "B_hi": 250, "I_lo": 101, "I_hi": 200},
        {"B_lo": 251, "B_hi": 350, "I_lo": 201, "I_hi": 300},
        {"B_lo": 351, "B_hi": 430, "I_lo": 301, "I_hi": 400},
        {"B_lo": 431, "B_hi": 600, "I_lo": 401, "I_hi": 500},
    ],
    "NO2": [
        {"B_lo": 0, "B_hi": 40, "I_lo": 0, "I_hi": 50},
        {"B_lo": 41, "B_hi": 80, "I_lo": 51, "I_hi": 100},
        {"B_lo": 81, "B_hi": 180, "I_lo": 101, "I_hi": 200},
        {"B_lo": 181, "B_hi": 280, "I_lo": 201, "I_hi": 300},
        {"B_lo": 281, "B_hi": 400, "I_lo": 301, "I_hi": 400},
        {"B_lo": 401, "B_hi": 500, "I_lo": 401, "I_hi": 500},
    ],
    "SO2": [
        {"B_lo": 0, "B_hi": 40, "I_lo": 0, "I_hi": 50},
        {"B_lo": 41, "B_hi": 80, "I_lo": 51, "I_hi": 100},
        {"B_lo": 81, "B_hi": 380, "I_lo": 101, "I_hi": 200},
        {"B_lo": 381, "B_hi": 800, "I_lo": 201, "I_hi": 300},
        {"B_lo": 801, "B_hi": 1600, "I_lo": 301, "I_hi": 400},
        {"B_lo": 1601, "B_hi": 2000, "I_lo": 401, "I_hi": 500},
    ],
    "CO": [
        {"B_lo": 0, "B_hi": 1, "I_lo": 0, "I_hi": 50},
        {"B_lo": 1.1, "B_hi": 2, "I_lo": 51, "I_hi": 100},
        {"B_lo": 2.1, "B_hi": 10, "I_lo": 101, "I_hi": 200},
        {"B_lo": 10.1, "B_hi": 17, "I_lo": 201, "I_hi": 300},
        {"B_lo": 17.1, "B_hi": 34, "I_lo": 301, "I_hi": 400},
        {"B_lo": 34.1, "B_hi": 50, "I_lo": 401, "I_hi": 500},
    ],
    "OZONE": [
        {"B_lo": 0, "B_hi": 50, "I_lo": 0, "I_hi": 50},
        {"B_lo": 51, "B_hi": 100, "I_lo": 51, "I_hi": 100},
        {"B_lo": 101, "B_hi": 168, "I_lo": 101, "I_hi": 200},
        {"B_lo": 169, "B_hi": 208, "I_lo": 201, "I_hi": 300},
        {"B_lo": 209, "B_hi": 748, "I_lo": 301, "I_hi": 400},
        {"B_lo": 749, "B_hi": 1000, "I_lo": 401, "I_hi": 500},
    ]
}


# **Final AQI Calculation and Categorization**

In [ ]:
# Takes all pollutants of a city and computes overall AQI
def calculate_aqi_cpcb(pollution_data):
    sub_indices = []

    # Loop through pollutants
    for pollutant, value in pollution_data.items():
        try:
            C = float(value) #Ensures numeric calculation
            bp = breakpoints.get(pollutant) #Fetches AQI ranges for that pollutant

            if bp:
                sub_index = calculate_sub_index(C, bp) #Converts pollutant → AQI scale
                if sub_index is not None:
                    sub_indices.append(sub_index) #Store values

        except ValueError: # Catch specific error for non-numeric values
            continue

    # CPCB Validity Check: Requires at least 3 pollutants, and one must be PM2.5 or PM10
    is_pm_present = 'PM2.5' in pollution_data or 'PM10' in pollution_data
    # We count pollutants for which a sub-index could be calculated
    num_valid_pollutants = len(sub_indices)

    if not is_pm_present or num_valid_pollutants < 3:
        return None, "Insufficient Data (CPCB requires PM2.5/PM10 and >=3 pollutants)"

    if not sub_indices:
        return None, "No valid pollutant data for AQI calculation"

    # Final AQI calculation
    aqi_value = round(max(sub_indices)) #Takes maximum sub-index
    category = get_aqi_category(aqi_value)

    return aqi_value, category

# Converts AQI value → human-readable category
def get_aqi_category(aqi):
    if aqi <= 50: return "Good "
    elif aqi <= 100: return "Satisfactory "
    elif aqi <= 200: return "Moderate "
    elif aqi <= 300: return "Poor "
    elif aqi <= 400: return "Very Poor "
    else: return "Severe "

In [ ]:
# Make the last_update variable globally accessible
GLOBAL_FORMATTED_API_UPDATED_DATE = last_update

# **Extract Pollution Data + Compute AQI (City-wise)**

In [ ]:
import requests

def get_pollution_data(selected_city):
    expected_pollutant_keys = {"CO", "SO2", "PM2.5", "PM10", "OZONE", "NO2"}

    city_pollution_data = {}
    station_uid = None

    # First, find a station in the selected city to get its UID
    for s in stations:
        if 'station' in s and 'name' in s['station']:
            station_name = s['station']['name']

            parts = station_name.split(',')
            api_city_name = parts[1].strip() if len(parts) > 1 else station_name.strip()

            if api_city_name == selected_city:
                station_uid = s['uid']
                break # Found a station, break the loop

    if station_uid:
        # Make a new API call to get detailed data for the specific station UID
        detailed_url = f"https://api.waqi.info/feed/@{station_uid}/?token={TOKEN}"
        detailed_response = requests.get(detailed_url)
        detailed_data = detailed_response.json()

        if detailed_data['status'] == 'ok' and 'data' in detailed_data:
            if 'iaqi' in detailed_data['data']:
                for pollutant_key, pollutant_value_obj in detailed_data['data']['iaqi'].items():
                    if pollutant_key.upper() in expected_pollutant_keys:
                        try:
                            # The value object usually contains a 'v' key for the value
                            city_pollution_data[pollutant_key.upper()] = float(pollutant_value_obj['v'])
                        except (ValueError, KeyError):
                            continue

    aqi_result, category_result = calculate_aqi_cpcb(city_pollution_data)
    global GLOBAL_FORMATTED_API_UPDATED_DATE
    api_metadata_date_for_return = GLOBAL_FORMATTED_API_UPDATED_DATE if 'GLOBAL_FORMATTED_API_UPDATED_DATE' in globals() else "N/A"


    return {
        "pollutants": city_pollution_data,
        "aqi": aqi_result,
        "category": category_result,
        "api_metadata_updated_date": api_metadata_date_for_return # Add the API metadata date here
    }

In [ ]:
def get_pollution_data_by_station_id(selected_city):
    expected_pollutant_keys = {"CO", "SO2", "PM2.5", "PM10", "OZONE", "NO2"}
    city_pollution_data = {}
    station_uid = district_station_ids.get(selected_city)

    if station_uid:
        # Make an API call to get detailed data for the specific station UID
        detailed_url = f"https://api.waqi.info/feed/{station_uid}/?token={TOKEN}"
        detailed_response = requests.get(detailed_url)
        detailed_data = detailed_response.json()

        if detailed_data['status'] == 'ok' and 'data' in detailed_data:
            if 'iaqi' in detailed_data['data']:
                for pollutant_key, pollutant_value_obj in detailed_data['data']['iaqi'].items():
                    # WAQI API sometimes uses 'pm25' instead of 'PM2.5' or 'o3' instead of 'OZONE'
                    standardized_key = pollutant_key.upper()
                    if standardized_key == 'PM25':
                        standardized_key = 'PM2.5'
                    elif standardized_key == 'O3':
                        standardized_key = 'OZONE'

                    if standardized_key in expected_pollutant_keys:
                        try:
                            city_pollution_data[standardized_key] = float(pollutant_value_obj['v'])
                        except (ValueError, KeyError):
                            continue

    aqi_result, category_result = calculate_aqi_cpcb(city_pollution_data)
    global GLOBAL_FORMATTED_API_UPDATED_DATE
    api_metadata_date_for_return = GLOBAL_FORMATTED_API_UPDATED_DATE if 'GLOBAL_FORMATTED_API_UPDATED_DATE' in globals() else "N/A"

    return {
        "pollutants": city_pollution_data,
        "aqi": aqi_result,
        "category": category_result,
        "api_metadata_updated_date": api_metadata_date_for_return
    }

In [ ]:
sample_station_uid = district_station_ids['Bengaluru']

sample_detailed_url = f"https://api.waqi.info/feed/{sample_station_uid}/?token={TOKEN}"
sample_detailed_response = requests.get(sample_detailed_url)
sample_detailed_data = sample_detailed_response.json()

if sample_detailed_data['status'] == 'ok' and 'data' in sample_detailed_data:
    print(f"Pollutant keys for {sample_station_uid}:")
    if 'iaqi' in sample_detailed_data['data']:
        for key in sample_detailed_data['data']['iaqi'].keys():
            print(f"- {key}")
    else:
        print("No 'iaqi' data found for this station.")
else:
    print(f"Error fetching data for station {sample_station_uid}: {sample_detailed_data.get('data', 'N/A')}")

In [ ]:
bagalkot_aqi_data = get_pollution_data_by_station_id("Bagalkot")

print(f"\nAQI Data for Bagalkot (using Station ID):")
print(f"Overall AQI: {bagalkot_aqi_data['aqi']}")
print(f"Category: {bagalkot_aqi_data['category']}")
print(f"Pollutants Data: {bagalkot_aqi_data['pollutants']}")
print(f"API Metadata Updated: {bagalkot_aqi_data['api_metadata_updated_date']}")

# **Map + Dashboard Rendering (Final Visualization Layer)**

In [ ]:
from IPython.display import display, HTML, clear_output
import json
import pandas as pd
import folium
from folium import plugins
from branca.element import Element, Figure

# ============================================================
# GLOBAL VARIABLE
# ============================================================

last_prediction_results = None

# ============================================================
# POLLUTANT INFO
# ============================================================

pollutant_causes_data = {
    "PM2.5": "Causes: Vehicle exhaust, industrial emissions. Effects: Respiratory and cardiovascular issues.",
    "PM10": "Causes: Dust, vehicle exhaust. Effects: Respiratory problems, irritation.",
    "NO2": "Causes: Vehicle exhaust, power plants. Effects: Respiratory illnesses, smog.",
    "SO2": "Causes: Burning fossil fuels, industrial processes. Effects: Respiratory problems, acid rain.",
    "CO": "Causes: Incomplete combustion of fuels. Effects: Reduces oxygen delivery, can be fatal.",
    "OZONE": "Causes: Chemical reactions in sunlight. Effects: Respiratory irritation, lung damage."
}

# ============================================================
# AQI COLOR FUNCTION
# ============================================================

def get_aqi_info_color(aqi):

    if aqi <= 50:
        return ["#00A651", "Good"]

    elif aqi <= 100:
        return ["#A3D977", "Satisfactory"]

    elif aqi <= 200:
        return ["#FFF200", "Moderate"]

    elif aqi <= 300:
        return ["#F7941D", "Poor"]

    elif aqi <= 400:
        return ["#ED1C24", "Very Poor"]

    else:
        return ["#7E0023", "Severe"]

# ============================================================
# REAL-TIME MAP
# ============================================================

def render_realtime_map(selected_city):

    if selected_city is None:
        display(HTML("<h2>No valid cities available.</h2>"))
        return

    data = get_pollution_data_by_station_id(selected_city)

    aqi = data["aqi"]
    category = data["category"]
    pollutants = data["pollutants"]
    api_metadata_updated_date = data["api_metadata_updated_date"]

    formatted_last_update = api_metadata_updated_date

    coords = city_coords.get(selected_city)

    if not coords:
        display(HTML(f"<h2>Coordinates not found for {selected_city}</h2>"))
        return

    # ========================================================
    # PROMINENT POLLUTANT
    # ========================================================

    prominent_pollutant = ""
    max_pollutant_value = -1

    for p, v in pollutants.items():

        try:
            float_v = float(v)

            if float_v > max_pollutant_value:
                max_pollutant_value = float_v
                prominent_pollutant = p

        except:
            continue

    prominent_cause_effect = pollutant_causes_data.get(
        prominent_pollutant,
        "Information not available."
    )

    # ========================================================
    # AQI COLOR
    # ========================================================

    aqi_color = "#AAAAAA"

    if aqi is not None:
        aqi_color = get_aqi_info_color(aqi)[0]

    # ========================================================
    # FIGURE
    # ========================================================

    fig = Figure(height="500px")

    # ========================================================
    # MAP
    # ========================================================

    m = folium.Map(
        location=coords,
        zoom_start=10,

        # CHANGED HERE
        tiles="CartoDB Voyager",

        attr="© CartoDB",
        zoom_control=False,
        scrollWheelZoom=False,
        dragging=False
    )

    # ========================================================
    # MARKER
    # ========================================================

    folium.CircleMarker(
        location=coords,
        radius=15,
        color=aqi_color,
        fill=True,
        fill_color=aqi_color,
        fill_opacity=0.5,
        weight=4,
        popup=f"{selected_city}<br>AQI: {aqi}<br>Category: {category}"
    ).add_to(m)

    # ========================================================
    # POLLUTANT CARDS
    # ========================================================

    pollutant_cards_html = ""

    for p in sorted(pollutants.keys()):

        v = pollutants[p]

        pollutant_cards_html += f"""
        <div class="pollutant-card">
            <span>{p}</span>
            <span>{round(float(v),2) if isinstance(v,(int,float)) else 'N/A'}</span>
        </div>
        """

    # ========================================================
    # WHITE UI STYLE
    # ========================================================

    style = f"""
    <style>

    body {{
        margin:0;
        font-family: Arial;
    }}

    .dashboard-container {{
        position: fixed;
        top: 10px;
        right: 10px;
        width: 320px;

        background-color: white;
        color: #111111;

        z-index: 9999;

        border-radius: 12px;
        padding: 15px;

        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;

        box-shadow: 0 0 20px rgba(0,0,0,0.15);

        border: 1px solid #dddddd;
    }}

    .header {{
        text-align: center;
        border-bottom: 1px solid #dddddd;
        padding-bottom: 8px;
    }}

    .aqi-box {{
        font-size: 32px;
        color: {aqi_color};
        font-weight: bold;
        margin: 12px 0;
        text-align: center;
    }}

    .pollutant-grid {{
        display: grid;
        grid-template-columns: 1fr 1fr;
        gap: 8px;
        margin-top: 12px;
    }}

    .pollutant-card {{
        background: #f5f5f5;
        padding: 8px;
        border-radius: 8px;
        font-size: 13px;
        display: flex;
        justify-content: space-between;
        color: #111111;
        border: 1px solid #e0e0e0;
    }}

    .scale-bar {{
        height: 8px;

        background: linear-gradient(
            to right,
            #00A651,
            #A3D977,
            #FFF200,
            #F7941D,
            #ED1C24,
            #7E0023
        );

        border-radius: 4px;
        margin: 10px 0;
    }}

    .leaflet-control-container {{
        display: none;
    }}

    </style>

    <div class="dashboard-container">

        <div class="header">

            <div style="font-size:10px; letter-spacing:1px;">
                CURRENT DATA
            </div>

            <div style="font-size:14px;">
                {formatted_last_update}
            </div>

            <div style="font-weight:bold; font-size:18px;">
                {selected_city.upper()}
            </div>

        </div>

        <div class="aqi-box">
            AQI: {aqi if aqi is not None else 'N/A'}
        </div>

        <div class="scale-bar"></div>

        <div style="
            font-size:12px;
            text-align:center;
            color:#222;
            font-weight:600;
        ">
            Prominent Pollutant: {prominent_pollutant} 🌁
        </div>

        <div class="pollutant-grid">
            {pollutant_cards_html}
        </div>

        <div style="
            font-size:11px;
            margin-top:15px;
            color:#444;
            border-top:1px solid #dddddd;
            padding-top:8px;
        ">

            <b>Cause & Effect:</b>

            {prominent_cause_effect}

        </div>

    </div>
    """

    fig.add_child(m)
    fig.get_root().html.add_child(Element(style))

    display(fig)

# ============================================================
# PREDICTED MAP
# ============================================================

def render_predicted_map(
    selected_city,
    pred,
    cat,
    trend,
    model_name,
    all_preds_dict,
    u_date
):

    if selected_city is None:
        display(HTML("<h2>No valid cities available.</h2>"))
        return

    current_data = get_pollution_data_by_station_id(selected_city)

    current_pollutants = current_data["pollutants"]

    formatted_user_date = "N/A"

    if u_date != "N/A":

        try:

            dt_object_user_date = pd.to_datetime(
                u_date,
                format='%Y-%m-%d'
            )

            formatted_user_date = dt_object_user_date.strftime('%d-%m-%Y')

        except:
            pass

    coords = city_coords.get(selected_city)

    if not coords:
        display(HTML(f"<h2>Coordinates not found for {selected_city}</h2>"))
        return

    # ========================================================
    # PROMINENT POLLUTANT
    # ========================================================

    prominent_pollutant = ""
    max_pollutant_value = -1

    for p, v in current_pollutants.items():

        try:

            float_v = float(v)

            if float_v > max_pollutant_value:
                max_pollutant_value = float_v
                prominent_pollutant = p

        except:
            continue

    prominent_cause_effect = pollutant_causes_data.get(
        prominent_pollutant,
        "Information not available."
    )

    # ========================================================
    # AQI COLOR
    # ========================================================

    aqi_color = "#AAAAAA"

    if pred is not None:
        aqi_color = get_aqi_info_color(pred)[0]

    # ========================================================
    # FIGURE
    # ========================================================

    fig = Figure(height="500px")

    # ========================================================
    # MAP
    # ========================================================

    m = folium.Map(
        location=coords,
        zoom_start=10,

        # CHANGED HERE
        tiles="CartoDB Voyager",

        attr="© CartoDB",

        zoom_control=False,
        scrollWheelZoom=False,
        dragging=False
    )

    # ========================================================
    # MARKER
    # ========================================================

    folium.CircleMarker(
        location=coords,
        radius=15,
        color=aqi_color,
        fill=True,
        fill_color=aqi_color,
        fill_opacity=0.5,
        weight=4,
        popup=f"{selected_city}<br>Predicted AQI: {pred}<br>Category: {cat}"
    ).add_to(m)

    # ========================================================
    # POLLUTANT CARDS
    # ========================================================

    pollutant_cards_html = ""

    for p in sorted(current_pollutants.keys()):

        v = current_pollutants[p]

        pollutant_cards_html += f"""
        <div class="pollutant-card">
            <span>{p}</span>
            <span>{round(float(v),2) if isinstance(v,(int,float)) else 'N/A'}</span>
        </div>
        """

    # ========================================================
    # STYLE
    # ========================================================

    style = f"""
    <style>

    body {{
        margin:0;
        font-family: Arial;
    }}

    .dashboard-container {{
        position: fixed;
        top: 10px;
        right: 10px;
        width: 320px;

        background-color: white;
        color: #111111;

        z-index: 9999;

        border-radius: 12px;
        padding: 15px;

        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;

        box-shadow: 0 0 20px rgba(0,0,0,0.15);

        border: 1px solid #dddddd;
    }}

    .header {{
        text-align: center;
        border-bottom: 1px solid #dddddd;
        padding-bottom: 8px;
    }}

    .aqi-box {{
        font-size: 32px;
        color: {aqi_color};
        font-weight: bold;
        margin: 12px 0;
        text-align: center;
    }}

    .pollutant-grid {{
        display: grid;
        grid-template-columns: 1fr 1fr;
        gap: 8px;
        margin-top: 12px;
    }}

    .pollutant-card {{
        background: #f5f5f5;
        padding: 8px;
        border-radius: 8px;
        font-size: 13px;
        display: flex;
        justify-content: space-between;
        color: #111111;
        border: 1px solid #e0e0e0;
    }}

    .scale-bar {{
        height: 8px;

        background: linear-gradient(
            to right,
            #00A651,
            #A3D977,
            #FFF200,
            #F7941D,
            #ED1C24,
            #7E0023
        );

        border-radius: 4px;
        margin: 10px 0;
    }}

    .leaflet-control-container {{
        display: none;
    }}

    </style>

    <div class="dashboard-container">

        <div class="header">

            <div style="font-size:10px; letter-spacing:1px;">
                PREDICTION FOR {formatted_user_date}
            </div>

            <div style="font-size:14px;">
                {model_name}
            </div>

            <div style="font-weight:bold; font-size:18px;">
                {selected_city.upper()}
            </div>

        </div>

        <div class="aqi-box">
            AQI: {pred if pred is not None else 'N/A'}
        </div>

        <div class="scale-bar"></div>

        <div style="
            font-size:12px;
            text-align:center;
            color:#222;
            font-weight:600;
        ">
            Prominent Pollutant: {prominent_pollutant} 🌁
        </div>

        <div class="pollutant-grid">
            {pollutant_cards_html}
        </div>

        <div style="
            font-size:11px;
            margin-top:15px;
            color:#444;
            border-top:1px solid #dddddd;
            padding-top:8px;
        ">

            <b>Cause & Effect:</b>

            {prominent_cause_effect}

        </div>

    </div>
    """

    fig.add_child(m)
    fig.get_root().html.add_child(Element(style))

    display(fig)

In [ ]:
render_realtime_map(dropdown.value)

# **Load Historical Dataset from Google Drive**

In [ ]:
from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
drive.mount('/content/drive')

folder_path = "/content/drive/My Drive/Dataset_ml"

all_data = [] # This list will hold individual dataframes

files = [f for f in os.listdir(folder_path) if f.endswith(('.xlsx', '.csv'))]

print("Files found:", files)

for file in files:
    print(f"Processing: {file}")
    try:
        # load
        if file.endswith(".xlsx"):
            df = pd.read_excel(os.path.join(folder_path, file))
        else:
            df = pd.read_csv(os.path.join(folder_path, file))

        # add city name
        city_name = file.split(".")[0]
        df["City"] = city_name

        all_data.append(df)
    except Exception as e:
        print(f"Error processing {file}: {e}")
        continue

print("\nAll individual city dataframes loaded into 'all_data' list.")
print(f"Number of dataframes loaded: {len(all_data)}")
# Display the head of the first dataframe to show the structure
if all_data:
    print("Head of the first loaded dataframe:")
    display(all_data[0].head())

# **shape and head of single data file**

In [ ]:
print("Shape of each DataFrame in 'all_data':")
for i, df_item in enumerate(all_data):
    if 'City' in df_item.columns:
        city_name = df_item['City'].iloc[0]
        print(f"  {city_name}: {df_item.shape}")
    else:
        print(f"  DataFrame {i+1}: {df_item.shape} (City name not found)")

# **fixing column and date format**

In [ ]:
def standardize_column_names(df):
    new_columns = {}
    for col in df.columns:
        # Convert to uppercase and replace '.' with '_' to standardize
        standardized_col = col.upper().replace('.', '_')
        new_columns[col] = standardized_col
    return df.rename(columns=new_columns)

def prepare_dataframe(df_item):
    df_item = standardize_column_names(df_item.copy())

    # FIXED DATE
    df_item['DATE'] = pd.to_datetime(df_item['DATE'], dayfirst=True, errors='coerce')

    # OPTIONAL: standard format
    df_item['DATE'] = df_item['DATE'].dt.strftime('%Y-%m-%d')
    df_item['DATE'] = pd.to_datetime(df_item['DATE'])

    if 'PM2_5' in df_item.columns:
        df_item.rename(columns={'PM2_5': 'PM2.5'}, inplace=True)

    return df_item

# Applying the `prepare_dataframe` function to clean each
# individual city's DataFrame and storing them in `cleaned_dataframes`.

cleaned_dataframes = []
for df_item in all_data:
    cleaned_df_item = prepare_dataframe(df_item)
    cleaned_dataframes.append(cleaned_df_item)

print("All dataframes in `cleaned_dataframes` are now cleaned and have standardized column names.")

# Display the head of the first cleaned dataframe to verify
if cleaned_dataframes:
    print("Head of the first cleaned dataframe:")
    display(cleaned_dataframes[0].head())

# **Identify Missing Values in Pollutant Columns**

In [ ]:
pollutant_columns = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'OZONE']

print("Summary of Missing Values (NaN) per Pollutant in Each City DataFrame:")
print("------------------------------------------------------------------")

for i, df_item in enumerate(cleaned_dataframes):
    city_name = df_item['CITY'].iloc[0] if 'CITY' in df_item.columns and not df_item['CITY'].empty else f"DataFrame {i+1}"

    print(f"\nCity: {city_name}")
    missing_counts = {}
    for col in pollutant_columns:
        if col in df_item.columns:
            # Count NaNs in the pollutant column
            missing_counts[col] = df_item[col].isna().sum()
        else:
            missing_counts[col] = "Column Not Found"

    for pollutant, count in missing_counts.items():
        print(f"  {pollutant}: {count}")


# **Fill Missing Pollutant Values**

In [ ]:
pollutant_columns = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'OZONE']

filled_dataframes = []

for i, df_item in enumerate(cleaned_dataframes):
    df_copy = df_item.copy()
    for col in pollutant_columns:
        if col in df_copy.columns:
            mean_value = df_copy[col].mean()
            df_copy[col] = df_copy[col].fillna(mean_value)
    filled_dataframes.append(df_copy)

In [ ]:
print("Shape of each DataFrame in 'filled_dataframes' after imputation:")
for i, df_item in enumerate(filled_dataframes):
    if 'CITY' in df_item.columns:
        city_name = df_item['CITY'].iloc[0]
        print(f"  {city_name}: {df_item.shape}")
    else:
        print(f"  DataFrame {i+1}: {df_item.shape} (City name not found)")

# **Total Records Across All Filled DataFrames**

In [ ]:
total_historical_records = 0
for df_item in filled_dataframes:
    total_historical_records += df_item.shape[0]

print(f"Total historical records across all cities (sum of rows from individual DataFrames): {total_historical_records}")

In [ ]:
# Display the raw data for the first city
print("Raw data (first city, before cleaning column names and date):")
display(all_data[0].head())

# Display the data for the same city after standardizing column names and date format
print("\nData after standardizing column names and date format (first city):")
display(cleaned_dataframes[0].head())

# **Calculate AQI for Historical Data**

In [ ]:
aqi_calculated_dataframes = []
pollutant_columns = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'OZONE'] # Ensure this matches the AQI breakpoints

print("Calculating AQI for each city's historical data...")
print("----------------------------------------------------")

for i, df_item in enumerate(filled_dataframes):
    df_copy = df_item.copy()
    city_name = df_copy['CITY'].iloc[0] if 'CITY' in df_copy.columns and not df_copy['CITY'].empty else f"DataFrame {i+1}"
    print(f"Processing AQI for: {city_name}")

    # Convert pollutant columns to numeric, coercing errors to NaN
    for col in pollutant_columns:
        if col in df_copy.columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')

    # Drop rows where critical pollutant values might have become NaN after coercion
    df_copy.dropna(subset=pollutant_columns, inplace=True)

    # Calculate AQI for each row
    df_copy['AQI'] = df_copy.apply(
        lambda row: calculate_aqi_cpcb({
            p: row[p] for p in pollutant_columns if p in row.index
        }), axis=1
    )
    aqi_calculated_dataframes.append(df_copy)

print("\nAll dataframes processed for AQI calculation.")
print("The AQI-calculated dataframes are stored individually in the 'aqi_calculated_dataframes' list.")

# Display the head of each AQI-calculated dataframe
print("\nHead of each AQI-calculated dataframe:")
for i, df_item in enumerate(aqi_calculated_dataframes):
    city_name = df_item['CITY'].iloc[0] if 'CITY' in df_item.columns and not df_item['CITY'].empty else f"DataFrame {i+1}"
    print(f"\nCity: {city_name}")
    display(df_item.head())

**Lag** - a previous time step or delayed value

-  **past historical values added as new columns**
-  feature engineering nor created or augmented data
-  they help the model to see previous aqi values and pollutant trends

**how do they matter to model:**
-  they add **time memory** to model because these models **do not** understand sequence/time

**y this matters for aqi:**
- aqi today strongly depends on yesterday pollution
- so lag helps capture the rising trends , falling trends

**y lag needed for the model when model already learns historical data :**
- model **do not** understand "row order" , they treat rows independently .
- without lag the model doesnt know "this row happened after another row"

**y used :**
- model **do not** automatically remember previous rows
- **so creating the lag features then feeding into the model**
- **the model only learns patterns from the training row that is provided and lag features are how that provide time-history**

**Then Why Only 3 or 15 Days?**
- Because v choose **HOW MUCH** history to show the model.

**So Final Understanding**
- **YES:** the model learns from historical data.

- **BUT:** lag features are the mechanism used to PRESENT that historical data to the model.

- **Without lag:** XGBoost/CatBoost cannot understand sequence/time properly.

In [ ]:
# ============================================================
# AQI FULL PIPELINE (WITH MAE REDUCTION UPGRADE INTEGRATED)
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# ============================================================
# 1. COMBINE DATA
# ============================================================

df_master = pd.concat(aqi_calculated_dataframes, ignore_index=True)

print("Initial Shape:", df_master.shape)

df_master['DATE'] = pd.to_datetime(df_master['DATE'])
df_master = df_master.sort_values(['CITY', 'DATE'])

# ============================================================
# 2. CLEAN AQI
# ============================================================

df_master['AQI'] = df_master['AQI'].apply(
    lambda x: x[0] if isinstance(x, tuple) else x
)

df_master['AQI'] = pd.to_numeric(df_master['AQI'], errors='coerce')

print("AQI cleaned")

# ============================================================
# 3. RESAMPLING (FIX TIME GAPS)
# ============================================================

resampled = []

for city in df_master['CITY'].unique():

    city_df = df_master[df_master['CITY'] == city].copy()
    city_df = city_df.sort_values('DATE')

    city_df = city_df.set_index('DATE')
    city_df = city_df.resample('D').mean(numeric_only=True)

    city_df['CITY'] = city
    city_df = city_df.interpolate().ffill().bfill()

    city_df = city_df.reset_index()

    resampled.append(city_df)

df_master = pd.concat(resampled, ignore_index=True)

print("After resampling:", df_master.shape)

# ============================================================
# 4. ENCODING
# ============================================================

le = LabelEncoder()
df_master['CITY_ID'] = le.fit_transform(df_master['CITY'])

# ============================================================
# 5. TIME FEATURES
# ============================================================

df_master['MONTH'] = df_master['DATE'].dt.month
df_master['DAY_OF_WEEK'] = df_master['DATE'].dt.dayofweek

# ============================================================
# 6. POLLUTANTS
# ============================================================

pollutants = ['PM2.5','PM10','NO2','SO2','CO','OZONE']

for col in pollutants:
    df_master[col] = pd.to_numeric(df_master[col], errors='coerce')

# ============================================================
# 7. LAG FEATURES (15 DAYS)
# ============================================================

for i in range(1, 16):
    df_master[f'AQI_LAG_{i}'] = df_master.groupby('CITY')['AQI'].shift(i)

selected_lags = [
    'AQI_LAG_1','AQI_LAG_2','AQI_LAG_3','AQI_LAG_5',
    'AQI_LAG_7','AQI_LAG_10','AQI_LAG_14','AQI_LAG_15'
]

# ============================================================
# 8. ROLLING FEATURES
# ============================================================

# df_master['AQI_ROLL_MEAN_3D'] = df_master.groupby('CITY')['AQI'].transform(lambda x: x.rolling(3).mean())
df_master['AQI_ROLL_MEAN_3D'] = df_master.groupby('CITY')['AQI'].transform(lambda x: x.rolling(3).mean())
df_master.fillna(method='bfill', inplace=True)
df_master['AQI_ROLL_STD_3D'] = df_master.groupby('CITY')['AQI'].transform(lambda x: x.rolling(3).std())

# ============================================================
# 9. DERIVED FEATURES
# ============================================================

df_master['AQI_TREND'] = df_master.groupby('CITY')['AQI'].diff()
df_master['AQI_DIFF_1'] = df_master.groupby('CITY')['AQI'].diff(1)

# ============================================================
# 🔥 MAE REDUCTION STEP 1: LOG TRANSFORM (FIXED POSITION)
# ============================================================

print("\nApplying log transform to AQI...")

df_master['AQI'] = np.log1p(df_master['AQI'])

print("✔ Log transform applied")

# ============================================================
# ⚡ MAE REDUCTION STEP 2: REMOVE NOISE FEATURES
# ============================================================

drop_features = ['AQI_TREND', 'AQI_DIFF_1']

# ============================================================
# 🌿 MAE REDUCTION STEP 3: ADD 7-DAY ROLLING FEATURE
# ============================================================

df_master['AQI_ROLL_MEAN_7D'] = (
    df_master.groupby('CITY')['AQI']
    .transform(lambda x: x.rolling(7).mean())
)

df_master['AQI_ROLL_MEAN_7D'] = (
    df_master['AQI_ROLL_MEAN_7D']
    .bfill()
    .ffill()
)

# ============================================================
# 10. CLEANING
# ============================================================

df_master = df_master.ffill().bfill()
df_train_ready = df_master.dropna().reset_index(drop=True)

print("Final dataset:", df_train_ready.shape)

# ============================================================
# 11. FEATURES
# ============================================================

features = (
    pollutants +
    selected_lags +
    ['AQI_ROLL_MEAN_3D','AQI_ROLL_STD_3D'] +
    ['MONTH','DAY_OF_WEEK'] +
    ['CITY_ID'] +
    ['AQI_ROLL_MEAN_7D']
)

# remove noisy ones safely
features = [f for f in features if f not in drop_features]

target = 'AQI'

# ============================================================
# 12. TRAIN / TEST SPLIT
# ============================================================

X = df_train_ready[features]
y = df_train_ready[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

print("\nREADY ✔")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nFINAL FEATURES:", len(features))
print(features)

# AQI PREDICTION PIPELINE

##  1. DATA PREPARATION
- Combined AQI datasets from multiple cities into one dataset  
- Converted `DATE` column into proper datetime format  
- Sorted data by:
  - CITY  
  - DATE  

---

##  2. AQI DATA CLEANING
- Fixed AQI values that were stored as tuples  
- Converted AQI into numeric format  
- Handled missing/invalid values using NaN conversion  

---

##  3. TIME SERIES STRUCTURING
- Converted dataset into daily time series  
- Used resampling (`resample('D')`) for each city  
- Filled missing dates using:
  - Interpolation  
  - Forward fill  
  - Backward fill  

 Result: Continuous daily AQI data for each city  

---

##  4. ENCODING
- Converted CITY names into numerical values using Label Encoding  
- Created:
  - CITY_ID → numerical representation of each city  

---

##  5. TIME FEATURES
- MONTH → seasonal pollution pattern  
- DAY_OF_WEEK → weekday vs weekend variation  

---

##  6. POLLUTANT FEATURES
Used:
- PM2.5  
- PM10  
- NO2  
- SO2  
- CO  
- OZONE  

 These represent real-time air quality conditions  

---

##  7. LAG FEATURES
- AQI_LAG_1 → yesterday  
- AQI_LAG_2 → 2 days ago  
- AQI_LAG_3 → 3 days ago  
- AQI_LAG_5 → 5 days ago  
- AQI_LAG_7 → 7 days ago  
- AQI_LAG_10 → 10 days ago  
- AQI_LAG_14 → 14 days ago  
- AQI_LAG_15 → 15 days ago  

 Captures historical pollution trends  

---

##  8. ROLLING FEATURES
- AQI_ROLL_MEAN_3D → 3-day average  
- AQI_ROLL_STD_3D → 3-day variation  
- AQI_ROLL_MEAN_7D → 7-day trend  

 Removes noise + captures smooth trends  

---

##  9. DERIVED FEATURES
- AQI_TREND → increase/decrease trend  
- AQI_DIFF_1 → day-to-day difference  

---

##  10. LOG TRANSFORMATION
Applied: AQI = log(1 + AQI)


 Benefits:
- Reduces extreme spikes  
- Normalizes distribution  
- Improves model accuracy  
- Reduces MAE significantly  

---

##  11. NOISE REMOVAL
Removed:
- AQI_TREND  
- AQI_DIFF_1  

---

##  12. FINAL FEATURES
Includes:
- Pollutants  
- Lag features  
- Rolling features  
- Time features  
- City encoding  

 Total ~20 features  

---

##  13. TRAIN-TEST SPLIT
- 80% training  
- 20% testing  
- No shuffling (time-series preserved)  

---

# FINAL RESULT
- Strong time-series ML pipeline  
- Captures trends, seasonality, and pollution patterns  
- Achieved very low MAE after log transform  

In [ ]:

!pip install xgboost
!pip install lightgbm
!pip install catboost

# **MODEL TRAINING**

In [ ]:
# ============================================================
# 🚀 MODEL UPGRADE BLOCK (MAE REDUCTION VERSION)
# ============================================================

import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# ============================================================
# 📊 EVALUATION FUNCTION
# ============================================================

def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print("-" * 40)
    print("MAE :", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R2  :", round(r2, 4))

    return [name, mae, rmse, r2]

# ============================================================
# 🌳 1. DECISION TREE (BASELINE)
# ============================================================

dt = DecisionTreeRegressor(
    max_depth=12,
    min_samples_leaf=5,
    random_state=42
)

dt.fit(X_train, y_train)
pred_dt = dt.predict(X_test)

res_dt = evaluate("Decision Tree", y_test, pred_dt)

# ============================================================
# 🌲 2. RANDOM FOREST (STRONG BASE MODEL ⭐)
# ============================================================

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

res_rf = evaluate("Random Forest", y_test, pred_rf)

# ============================================================
# ⚡ 3. XGBOOST (MAE OPTIMIZED)
# ============================================================

xgb = XGBRegressor(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42
)

xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)

res_xgb = evaluate("XGBoost", y_test, pred_xgb)

# ============================================================
# 🌿 4. LIGHTGBM (FAST + STRONG)
# ============================================================

lgbm = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=8,
    num_leaves=64,
    min_child_samples=10,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    verbose=-1
)

lgbm.fit(X_train, y_train)
pred_lgbm = lgbm.predict(X_test)

res_lgbm = evaluate("LightGBM", y_test, pred_lgbm)

# ============================================================
# 🧠 5. CATBOOST (VERY STRONG FOR AQI DATA)
# ============================================================

cat = CatBoostRegressor(
    iterations=800,
    learning_rate=0.03,
    depth=8,
    loss_function='RMSE',
    random_seed=42,
    verbose=0
)

cat.fit(X_train, y_train)
pred_cat = cat.predict(X_test)

res_cat = evaluate("CatBoost", y_test, pred_cat)

# ============================================================
# 📊 FINAL COMPARISON TABLE
# ============================================================

results = [
    res_dt,
    res_rf,
    res_xgb,
    res_lgbm,
    res_cat
]

df_results = pd.DataFrame(results, columns=["Model", "MAE", "RMSE", "R2"])

In [ ]:
print("\n================ FINAL MODEL COMPARISON ================\n")
print(df_results.sort_values(by="MAE").round(3))
print(f"\n BEST MODEL : {df_results.sort_values(by="MAE").iloc[0]["Model"]}")

# **FEATURE IMPORTANCE**

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# FEATURE IMPORTANCE COMPARISON (ENSEMBLE MODELS)
# ============================================================

features_list = X_train.columns

rf_imp = pd.Series(rf.feature_importances_, index=features_list)
xgb_imp = pd.Series(xgb.feature_importances_, index=features_list)
lgbm_imp = pd.Series(lgbm.feature_importances_, index=features_list)
cat_imp = pd.Series(cat.get_feature_importance(), index=features_list)

# ------------------------------------------------------------
# NORMALIZATION FUNCTION (0–1 SCALE)
# ------------------------------------------------------------
def normalize(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

rf_imp = normalize(rf_imp)
xgb_imp = normalize(xgb_imp)
lgbm_imp = normalize(lgbm_imp)
cat_imp = normalize(cat_imp)

# ------------------------------------------------------------
# FINAL ENSEMBLE IMPORTANCE (AVERAGE OF ALL MODELS)
# ------------------------------------------------------------
final_importance = (
    rf_imp +
    xgb_imp +
    lgbm_imp +
    cat_imp
) / 4

final_importance = final_importance.sort_values(ascending=False)

print("\n🔥 FINAL FEATURE IMPORTANCE (ENSEMBLE AVERAGE)")
print(final_importance)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 7))
final_importance.plot(kind='bar')
plt.title('Final Feature Importance (Ensemble Average)')
plt.xlabel('Features')
plt.ylabel('Normalized Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**INVERSE LOG TRANSFORM (BACK TO ORIGINAL AQI SCALE)**

In [ ]:
# ============================================================
# CONVERT LOG AQI BACK TO ORIGINAL SCALE
# ============================================================

y_test_actual = np.expm1(y_test)
pred_rf_actual = np.expm1(pred_rf)

print("\n🔥 ACTUAL AQI SAMPLE:")
print(y_test_actual.head())

print("\n🔥 PREDICTED AQI SAMPLE:")
print(pred_rf_actual[:5])

# **Prediction Error Analysis**

In [ ]:
# ============================================================
# ACTUAL vs PREDICTED COMPARISON + ERROR
# ============================================================
comparison_df = pd.DataFrame({
    'Actual_AQI': y_test_actual.values,
    'Predicted_AQI': pred_rf_actual
})

# Calculate error
comparison_df['Error'] = (
    comparison_df['Actual_AQI'] -
    comparison_df['Predicted_AQI']
)

# Absolute Error
comparison_df['Absolute_Error'] = (
    comparison_df['Error'].abs()
)

print("\nAQI Prediction Comparison:")
display(comparison_df.head(10))

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(comparison_df["Actual_AQI"], comparison_df["Predicted_AQI"], alpha=0.6, s=20)
plt.plot([comparison_df["Actual_AQI"].min(), comparison_df["Actual_AQI"].max()],
         [comparison_df["Actual_AQI"].min(), comparison_df["Actual_AQI"].max()],
         'r--', lw=2, label='Perfect Prediction')
plt.title("Actual vs. Predicted AQI")
plt.xlabel("Actual AQI")
plt.ylabel("Predicted AQI")
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

# **Overall Model Performance Summary**

Model predicts AQI in log scale and is converted back using expm1 before display.


In [ ]:
from sklearn.metrics import mean_absolute_error

real_mae = mean_absolute_error(y_test_actual, pred_rf_actual)

print("\nREAL MAE (Original AQI Scale):", round(real_mae, 4))

**Direct one-step forecasting (Next-day AQI)**

In [ ]:
rf  # your best Random Forest model

# **City-wise AQI Prediction (Next-Day Forecast + Previous-Day Validation)**

In [ ]:
# ============================================================
# COMPLETE INTERACTIVE AQI PREDICTION SYSTEM
# RANDOM FOREST + REALTIME MAP + NEXT DAY PREDICTION
# ============================================================

import numpy as np
import pandas as pd
from datetime import timedelta

from ipywidgets import Dropdown, Button, VBox, Output
from IPython.display import display

# ============================================================
# DROPDOWN
# ============================================================

valid_cities = df_train_ready['CITY'].unique()

dropdown = Dropdown(
    options=sorted(valid_cities),
    description='City:'
)

output = Output()

# ============================================================
# NEXT DAY PREDICTION FUNCTION
# ============================================================

def predict_next_day(city_name):

    city_data = (
        df_train_ready[df_train_ready['CITY'] == city_name]
        .sort_values('DATE')
    )

    last_row = city_data.iloc[-1:].copy()

    current_input = last_row[X_train.columns]

    pred_log = rf.predict(current_input)[0]

    pred_aqi = np.expm1(pred_log)

    return pred_aqi

# ============================================================
# PREVIOUS DAY VALIDATION FUNCTION
# ============================================================

def predict_previous_day(city_name):

    city_data = (
        df_train_ready[df_train_ready['CITY'] == city_name]
        .sort_values('DATE')
    )

    input_row = city_data.iloc[-2:-1].copy()

    X_input = input_row[X_train.columns]

    pred_log = rf.predict(X_input)[0]

    pred_aqi = np.expm1(pred_log)

    actual_last = np.expm1(city_data.iloc[-1]['AQI'])

    error = abs(actual_last - pred_aqi)

    return actual_last, pred_aqi, error

# ============================================================
# BUTTON 1 : NEXT DAY AQI PREDICTION
# ============================================================

button_next = Button(
    description="Predict Next Day AQI"
)

def on_next_click(b):

    with output:

        output.clear_output()

        # ------------------------------------------------
        # SELECTED CITY
        # ------------------------------------------------

        city = dropdown.value

        # ------------------------------------------------
        # PREDICTION
        # ------------------------------------------------

        pred = predict_next_day(city)

        pred_display = int(round(pred))

        print(f"City: {city}")
        print(f"Next Day AQI Prediction: {pred_display}")

        # ------------------------------------------------
        # CATEGORY
        # ------------------------------------------------

        try:
            cat = get_aqi_info_color(pred_display)[1]
        except:
            cat = "Unknown"

        print(f"Category: {cat}")

        # ------------------------------------------------
        # NEXT DAY DATE
        # ------------------------------------------------

        next_day = (
            pd.Timestamp.today() + timedelta(days=1)
        ).strftime('%Y-%m-%d')

        print(f"Prediction Date: {next_day}")

        # ------------------------------------------------
        # MAP RENDERING
        # ------------------------------------------------

        try:

            render_predicted_map(
                selected_city=city,
                pred=pred_display,
                cat=cat,
                trend=None,
                model_name="Random Forest",
                all_preds_dict={},
                u_date=next_day
            )

        except Exception as e:

            print("Map Error:", e)

# Attach button event
button_next.on_click(on_next_click)

# ============================================================
# BUTTON 2 : PREVIOUS DAY ACCURACY CHECK
# ============================================================

button_prev = Button(
    description="Check Previous Day Accuracy"
)

def on_prev_click(b):

    with output:

        output.clear_output()

        city = dropdown.value

        actual, pred, error = predict_previous_day(city)

        print(f"City: {city}")

        print(f"Actual AQI (Last Day): {int(round(actual))}")

        print(f"Predicted AQI (Previous Day): {int(round(pred))}")

        print(f"Absolute Error: {round(error, 2)}")

# Attach button event
button_prev.on_click(on_prev_click)

# ============================================================
# DISPLAY UI
# ============================================================

display(
    VBox([
        dropdown,
        button_next,
        button_prev,
        output
    ])
)

# **ERROR ANALYSIS SYSTEM (CITY-WISE PERFORMANCE)**

In [ ]:
# ============================================================
# CITY-WISE ERROR ANALYSIS (REAL AQI SCALE)
# ============================================================

from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

city_errors = []

for city in df_train_ready['CITY'].unique():

    city_df = df_train_ready[df_train_ready['CITY'] == city].sort_values('DATE')

    X_city = city_df[X_train.columns]

    # ACTUAL AQI (convert back)
    y_city_actual = np.expm1(city_df['AQI'])

    # PREDICT (log scale → convert back)
    pred_log = rf.predict(X_city)
    pred_actual = np.expm1(pred_log)

    mae = mean_absolute_error(y_city_actual, pred_actual)

    city_errors.append([city, mae])

error_df = pd.DataFrame(city_errors, columns=['City', 'MAE'])
error_df = error_df.sort_values('MAE')

# ============================================================
# RESULTS
# ============================================================

print("\n📊 CITY-WISE ERROR ANALYSIS (REAL SCALE)\n")
print(error_df)

print("\n🏆 BEST CITY (Lowest Error):")
print(error_df.iloc[0])

print("\n⚠ WORST CITY (Highest Error):")
print(error_df.iloc[-1])

# ============================================================
# PLOT
# ============================================================

plt.figure(figsize=(10,5))
plt.bar(error_df['City'], error_df['MAE'])
plt.xticks(rotation=90)
plt.title("City-wise MAE Comparison (Real AQI Scale)")
plt.ylabel("MAE")
plt.show()

# **FEATURE vs AQI INSIGHT SYSTEM**

In [ ]:
# ============================================================
# FEATURE vs AQI INSIGHT SYSTEM (REAL SCALE)
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ----------------------------------------------------
# 1. FEATURE IMPORTANCE VISUALIZATION
# ----------------------------------------------------

final_importance.head(10).plot(kind='bar', figsize=(10,5))
plt.title("Top Features Affecting AQI")
plt.ylabel("Importance Score")
plt.show()

# ----------------------------------------------------
# 2. ACTUAL vs PREDICTED COMPARISON (REAL AQI SCALE)
# ----------------------------------------------------

y_test_actual = np.expm1(y_test)
pred_rf_actual = np.expm1(rf.predict(X_test))

comparison = pd.DataFrame({
    "Actual_AQI": y_test_actual.values,
    "Predicted_AQI": pred_rf_actual
})

# ----------------------------------------------------
# 3. ERROR CALCULATION
# ----------------------------------------------------

comparison["Error"] = comparison["Actual_AQI"] - comparison["Predicted_AQI"]
comparison["Absolute_Error"] = comparison["Error"].abs()

# ----------------------------------------------------
# 4. OUTPUT SAMPLE
# ----------------------------------------------------

print("\n📊 FEATURE vs AQI INSIGHT SYSTEM")
print("\nSample Predictions:")
print(comparison.head(10))

# **City + Date Based AQI Forecasting System (Recursive Hidden Simulation)**

In [ ]:
# ============================================================
# FUTURE AQI FORECAST SYSTEM (IMPROVED)
# ============================================================

import numpy as np
import pandas as pd
from ipywidgets import Dropdown, DatePicker, Button, VBox, Output
from IPython.display import display

# ----------------------------------------------------
# UI COMPONENTS
# ----------------------------------------------------

cities = df_train_ready['CITY'].unique()

city_dropdown = Dropdown(
    options=sorted(cities),
    description='City:'
)

date_picker = DatePicker(
    description='Target Date'
)

output = Output()

button = Button(description="Predict AQI")

# ----------------------------------------------------
# FORECAST ENGINE (FIXED RECURRENCE)
# ----------------------------------------------------

def forecast_to_date(city_name, target_date):

    city_data = df_train_ready[df_train_ready['CITY'] == city_name].sort_values('DATE')

    last_date = pd.to_datetime(city_data['DATE'].max())
    target_date = pd.to_datetime(target_date)

    current_row = city_data.iloc[-1:].copy()
    current_row = current_row[X_train.columns]

    days_ahead = (target_date - last_date).days

    if days_ahead <= 0:
        return np.expm1(rf.predict(current_row)[0])

    predictions = []

    for _ in range(days_ahead):

        X_input = current_row[X_train.columns]

        pred_log = rf.predict(X_input)[0]
        pred_aqi = np.expm1(pred_log)

        predictions.append(pred_aqi)

        # ------------------------------------------------
        # FIXED LAG SHIFT LOGIC
        # ------------------------------------------------

        new_row = current_row.copy()

        for lag in range(15, 1, -1):
            prev_col = f"AQI_LAG_{lag-1}"
            curr_col = f"AQI_LAG_{lag}"

            if prev_col in new_row.columns and curr_col in new_row.columns:
                new_row[curr_col] = new_row[prev_col]

        new_row["AQI_LAG_1"] = pred_log

        current_row = new_row

    return predictions[-1]

# ----------------------------------------------------
# BUTTON ACTION
# ----------------------------------------------------

def on_click(b):

    with output:
        output.clear_output()

        city = city_dropdown.value
        target_date = date_picker.value

        if target_date is None:
            print("⚠ Please select a date")
            return

        pred = forecast_to_date(city, target_date)

        pred_display = int(round(pred))

        print(f"City: {city}")
        print(f"Target Date: {target_date}")
        print(f"Predicted AQI: {pred_display:.2f}")

        # category
        try:
            cat = get_aqi_info_color(pred_display)[1]
        except:
            cat = "Unknown"

        # map
        try:
            render_predicted_map(
                selected_city=city,
                pred=pred_display,
                cat=cat,
                trend=None,
                model_name="Random Forest",
                all_preds_dict={},
                u_date=str(target_date)
            )
        except Exception as e:
            print("Map Error:", e)

button.on_click(on_click)

# ----------------------------------------------------
# DISPLAY UI
# ----------------------------------------------------

display(VBox([city_dropdown, date_picker, button, output]))
# display(VBox([city_dropdown, date_picker, button, output]))